<a href="https://colab.research.google.com/github/AnthonyMath1022/AnthonyMath1022/blob/main/favorita_ChebConvTCN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pytorch-forecasting torch-geometric pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.5.1+cu124.html

Looking in links: https://data.pyg.org/whl/torch-2.5.1+cu124.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 225.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 117.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 130.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.8/399.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4

In [2]:
!pip install kaggle pyunpack patool
from google.colab import files
files.upload()  # Upload your kaggle.json

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!kaggle competitions download favorita-grocery-sales-forecasting -p /content/favorita/
!unzip /content/favorita/favorita-grocery-sales-forecasting.zip -d /content/favorita/

from pyunpack import Archive
import glob
for f in glob.glob('/content/favorita/*.7z'):
    Archive(f).extractall('/content/favorita/')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 3.3 MB/s eta 0:00:00


Saving kaggle.json to kaggle.json
 69% 314M/458M [00:00<00:00, 3.29GB/s]
100% 458M/458M [00:00<00:00, 3.28GB/s]
Archive:  /content/favorita/favorita-grocery-sales-forecasting.zip
  inflating: /content/favorita/holidays_events.csv.7z  
  inflating: /content/favorita/items.csv.7z  
  inflating: /content/favorita/oil.csv.7z  
  inflating: /content/favorita/sample_submission.csv.7z  
  inflating: /content/favorita/stores.csv.7z  
  inflating: /content/favorita/test.csv.7z  
  inflating: /content/favorita/train.csv.7z  
  inflating: /content/favorita/transactions.csv.7z  


In [3]:
import pandas as pd, numpy as np, gc

dtype_dict = {'store_nbr': 'int16', 'item_nbr': 'int32',
              'unit_sales': 'float32', 'onpromotion': 'object'}

# Chunked reading with date filter — robust to row-index drift
chunks = []
for chunk in pd.read_csv('/content/favorita/train.csv',
                          usecols=['date','store_nbr','item_nbr','unit_sales','onpromotion'],
                          dtype=dtype_dict, parse_dates=['date'], chunksize=5_000_000):
    chunks.append(chunk[chunk['date'] >= '2014-07-01'])
train_df = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()
train_df['log_sales'] = np.log1p(train_df['unit_sales'].clip(lower=0))

In [4]:
# Load auxiliary files
stores = pd.read_csv('/content/favorita/stores.csv')
items = pd.read_csv('/content/favorita/items.csv')
oil = pd.read_csv('/content/favorita/oil.csv', parse_dates=['date'])
holidays = pd.read_csv('/content/favorita/holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('/content/favorita/transactions.csv', parse_dates=['date'])

# Oil: forward-fill missing weekend/holiday values
oil = oil.rename(columns={'dcoilwtico': 'oil'}).dropna(subset=['oil'])
oil = oil.set_index('date').resample('1D').ffill().reset_index()

# Holidays: split by locale, exclude transferred
holidays = holidays[~holidays['transferred']]
national = holidays[holidays['locale']=='National'][['date','description']].drop_duplicates('date')
national.columns = ['date','national_hol']
regional = holidays[holidays['locale']=='Regional'][['date','locale_name','description']].drop_duplicates(['date','locale_name'])
regional.columns = ['date','state','regional_hol']
local = holidays[holidays['locale']=='Local'][['date','locale_name','description']].drop_duplicates(['date','locale_name'])
local.columns = ['date','city','local_hol']

# Time features (known in advance)
train_df['day_of_week'] = train_df['date'].dt.dayofweek.astype(str)
train_df['day_of_month'] = train_df['date'].dt.day.astype('float32')
train_df['month'] = train_df['date'].dt.month.astype('float32')
train_df['open'] = 1.0  # All observed rows are open; filled rows get 0

# Merge everything
train_df = (train_df
    .merge(stores, on='store_nbr', how='left')
    .merge(items, on='item_nbr', how='left')
    .merge(oil, on='date', how='left')
    .merge(transactions, on=['date','store_nbr'], how='left')
    .merge(national, on='date', how='left')
    .merge(regional, on=['date','state'], how='left')
    .merge(local, on=['date','city'], how='left'))

# Fill NaN in merged columns
train_df['oil'] = train_df['oil'].ffill().bfill().astype('float32')
train_df['transactions'] = train_df['transactions'].fillna(0).astype('float32')
for col in ['national_hol','regional_hol','local_hol']:
    train_df[col] = train_df[col].fillna('None')

# Entity ID
train_df['traj_id'] = train_df['store_nbr'].astype(str) + '_' + train_df['item_nbr'].astype(str)

In [5]:
pip install torch-geometric

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np


# 1. Aggregate to the Store Level (54 Nodes)
# Sum sales/transactions, take the first value for global features like oil
store_df = train_df.groupby(['date', 'store_nbr']).agg({
    'log_sales': 'sum',
    'transactions': 'sum',
    'oil': 'first',
    'day_of_month': 'first'
}).reset_index()

# 2. Pivot to build the Store-to-Store Correlation Matrix
pivot_sales = store_df.pivot_table(index='date', columns='store_nbr', values='log_sales', fill_value=0)
node_corr_matrix = pivot_sales.corr().values

# 3. Build edge_index for PyTorch Geometric
threshold = 0.6
edge_indices, edge_weights = [], []
num_nodes = node_corr_matrix.shape[0]

for i in range(num_nodes):
    for j in range(num_nodes):
        if i != j and abs(node_corr_matrix[i, j]) > threshold:
            edge_indices.append([i, j])
            edge_weights.append(node_corr_matrix[i, j])

edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weights, dtype=torch.float)

print(f"Graph built: {num_nodes} nodes (stores) with {edge_index.shape[1]} edges.")

Graph built: 54 nodes (stores) with 2270 edges.


In [7]:
from sklearn.preprocessing import StandardScaler
import torch


# 1. Reshape features into [Total_Time, Nodes, Features]
features = ['log_sales', 'transactions', 'oil', 'day_of_month']
num_features = len(features)
total_time_steps = len(pivot_sales.index)

# Create an empty tensor to hold our structured data
# Shape: [Time, 54 Stores, 4 Features]
data_tensor = torch.zeros((total_time_steps, num_nodes, num_features))

for feature_idx, feature_name in enumerate(features):
    pivot_feat = store_df.pivot_table(index='date', columns='store_nbr', values=feature_name, fill_value=0)
    data_tensor[:, :, feature_idx] = torch.tensor(pivot_feat.values, dtype=torch.float32)

# The target is just log_sales (feature index 0)
target_tensor = data_tensor[:, :, 0]

# 2. Define the Custom Dataset
class FavoritaGraphDataset(Dataset):
    def __init__(self, data, target, window_size=90, horizon=30):
        self.data = data
        self.target = target
        self.window_size = window_size
        self.horizon = horizon

    def __len__(self):
        return len(self.data) - self.window_size - self.horizon + 1

    def __getitem__(self, idx):
        # Extract the lookback window
        # Shape: [Nodes, Time, Features] -> PyTorch standard for TCNs
        X = self.data[idx : idx + self.window_size].permute(1, 0, 2)

        # Extract the forecast horizon targets
        # Shape: [Nodes, Time]
        Y = self.target[idx + self.window_size : idx + self.window_size + self.horizon].permute(1, 0)

        return X, Y

from torch.utils.data import DataLoader

# 1. Define the cutoff points for the temporal split
train_end = int(total_time_steps * 0.70)
val_end = int(total_time_steps * 0.85)

# 1. Initialize the scaler (using StandardScaler to match the paper's z-score method)
scaler = StandardScaler()

# 2. Extract ONLY the training portion of the sales data (Feature 0)
# We have to reshape it to 2D because sklearn expects (samples, features)
train_sales_2d = data_tensor[:train_end, :, 0].reshape(-1, 1)

# 3. FIT the scaler exclusively on the training data
scaler.fit(train_sales_2d)

# 4. TRANSFORM all three sets using the training-fitted scaler
# We reshape to 2D, transform, and then reshape back to the [Time, 54 Stores] format

# Transform Train
data_tensor[:train_end, :, 0] = torch.tensor(
    scaler.transform(data_tensor[:train_end, :, 0].reshape(-1, 1)).reshape(-1, 54),
    dtype=torch.float32
)

# Transform Validation
data_tensor[train_end:val_end, :, 0] = torch.tensor(
    scaler.transform(data_tensor[train_end:val_end, :, 0].reshape(-1, 1)).reshape(-1, 54),
    dtype=torch.float32
)

# Transform Test
data_tensor[val_end:, :, 0] = torch.tensor(
    scaler.transform(data_tensor[val_end:, :, 0].reshape(-1, 1)).reshape(-1, 54),
    dtype=torch.float32
)

# 5. Update the target tensor to reflect the newly scaled values
target_tensor = data_tensor[:, :, 0]

# --- NOW you can define your Datasets and DataLoaders! ---
train_dataset = FavoritaGraphDataset(data_tensor[:train_end], target_tensor[:train_end])
val_dataset = FavoritaGraphDataset(data_tensor[train_end:val_end], target_tensor[train_end:val_end])
test_dataset = FavoritaGraphDataset(data_tensor[val_end:], target_tensor[val_end:])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. Slice the tensors for each dataset
train_dataset = FavoritaGraphDataset(
    data_tensor[:train_end], target_tensor[:train_end]
)
val_dataset = FavoritaGraphDataset(
    data_tensor[train_end:val_end], target_tensor[train_end:val_end]
)
test_dataset = FavoritaGraphDataset(
    data_tensor[val_end:], target_tensor[val_end:]
)

# 3. Create the PyTorch DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
# Note: We NEVER shuffle the validation or test loaders in time-series!

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from torch_geometric.typing import Adj, OptTensor
from torch_geometric.nn import ChebConv
import random
import os

class MultiHeadTemporalAttention(nn.Module):
    """
    Multi-head temporal attention with optional causal masking.
    Processes temporal sequences with multiple attention heads.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_heads: int = 8,
        dropout: float = 0.1,
        causal_mask: bool = False
    ):
        super(MultiHeadTemporalAttention, self).__init__()

        assert out_channels % num_heads == 0, \
            f"out_channels ({out_channels}) must be divisible by num_heads ({num_heads})"

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_heads = num_heads
        self.head_dim = out_channels // num_heads
        self.causal_mask = causal_mask

        self.query_proj = nn.Linear(in_channels, out_channels)
        self.key_proj = nn.Linear(in_channels, out_channels)
        self.value_proj = nn.Linear(in_channels, out_channels)
        self.out_proj = nn.Linear(out_channels, out_channels)

        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5

    def create_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """Create causal mask to prevent attending to future positions."""
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.bool()

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Args:
            x: [B, T, N, F_in] - Batch, Time, Nodes, Features
            attn_mask: [T, T] - Attention mask (True = masked)
            key_padding_mask: [B, T] - Padding mask (True = masked)

        Returns:
            out: [B, T, N, F_out]
        """
        B, T, N, F_in = x.shape

        # Reshape: [B, T, N, F] -> [B*N, T, F]
        x_reshaped = x.permute(0, 2, 1, 3).contiguous().view(B * N, T, F_in)

        # Project to Q, K, V
        Q = self.query_proj(x_reshaped)  # [B*N, T, out_channels]
        K = self.key_proj(x_reshaped)
        V = self.value_proj(x_reshaped)

        # Reshape for multi-head: [B*N, T, num_heads, head_dim] -> [B*N, num_heads, T, head_dim]
        Q = Q.view(B * N, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B * N, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B * N, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention scores: [B*N, num_heads, T, T]
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        # Apply causal mask
        if self.causal_mask:
            causal_mask = self.create_causal_mask(T, x.device)
            attn_scores = attn_scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        # Apply custom attention mask
        if attn_mask is not None:
            attn_scores = attn_scores.masked_fill(attn_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        # Apply key padding mask
        if key_padding_mask is not None:
            key_padding_mask = key_padding_mask.repeat_interleave(N, dim=0)
            key_padding_mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_scores = attn_scores.masked_fill(key_padding_mask, float('-inf'))

        # Softmax and dropout
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Apply attention to values
        attn_out = torch.matmul(attn_weights, V)  # [B*N, num_heads, T, head_dim]
        attn_out = attn_out.transpose(1, 2).contiguous().view(B * N, T, self.out_channels)

        # Output projection
        out = self.out_proj(attn_out)  # [B*N, T, out_channels]

        # Reshape back: [B*N, T, F] -> [B, T, N, F]
        out = out.view(B, N, T, self.out_channels).permute(0, 2, 1, 3)

        return out


class TemporalConvNet(nn.Module):
    """Temporal Convolutional Network with causal convolutions."""
    def __init__(
        self,
        in_channels: int,
        hidden_channels: list,
        kernel_size: int = 10,
        dropout: float = 0.2
    ):
        super().__init__()

        layers = []
        num_levels = len(hidden_channels)

        for i in range(num_levels):
            in_ch = in_channels if i == 0 else hidden_channels[i - 1]
            out_ch = hidden_channels[i]
            dilation = 2 ** i
            # Causal padding: only pad on the left side
            padding = (kernel_size - 1) * dilation

            layers.append(
                nn.Conv1d(
                    in_ch, out_ch, kernel_size,
                    padding=padding,
                    dilation=dilation
                )
            )
            layers.append(nn.Tanh())
            layers.append(nn.Dropout(dropout))

        self.network = nn.Sequential(*layers)
        self.kernel_size = kernel_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, C, T] input tensor

        Returns:
            out: [B, C, T] output with same temporal length (causal)
        """
        out = self.network(x)
        # Crop to maintain causality and original length
        return out[:, :, :x.size(2)]


class ChebConvTCN(nn.Module):
    """
    Chebyshev Convolutional Temporal Graph Network with Multi-Head Attention.

    Flow: ChebConv -> Multi-Head Temporal Attention -> TCN -> Output
    """
    def __init__(
        self,
        in_channels: int,
        g_hidden: int,
        spatial_channels: int,
        temporal_channels: list,
        forecast_horizon: int = 5,
        K: int = 5,
        kernel_size: int = 8,
        dropout: float = 0.2,
        use_temporal_attention: bool = True,
        num_heads: int = 8,
        attn_dropout: float = 0.1,
        attn_residual: bool = True,
        use_layernorm_after_attn: bool = True,
        causal_mask: bool = False,
        quantiles: list = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
    ):
        super().__init__()

        self.in_channels = in_channels
        self.spatial_channels = spatial_channels
        self.use_temporal_attention = use_temporal_attention
        self.attn_residual = attn_residual
        self.forecast_horizon = forecast_horizon
        self.quantiles = quantiles                    # NEW
        self.num_quantiles = len(quantiles)

        # Spatial graph convolution (ChebConv)
        self.gconv = ChebConv(in_channels, g_hidden, K)

        # Projection layer to match dimensions
        self.spatial_proj = nn.Linear(g_hidden, spatial_channels)

        # Residual connection for input
        self.residual = nn.Linear(in_channels, g_hidden) if in_channels != g_hidden else nn.Identity()

        # Multi-head temporal attention
        if use_temporal_attention:
            self.temporal_attn = MultiHeadTemporalAttention(
                in_channels=spatial_channels,
                out_channels=spatial_channels,
                num_heads=num_heads,
                dropout=attn_dropout,
                causal_mask=causal_mask
            )
        else:
            self.temporal_attn = None

        # Layer normalization after attention
        self.norm_after_attn = (
            nn.LayerNorm(spatial_channels)
            if (use_temporal_attention and use_layernorm_after_attn)
            else None
        )

        # Temporal convolutional network
        self.temporal_conv = TemporalConvNet(
            in_channels=spatial_channels,
            hidden_channels=temporal_channels,
            kernel_size=kernel_size,
            dropout=dropout
        )

        # Output projection
        self.output_layer = nn.Linear(
            temporal_channels[-1],
            forecast_horizon * self.num_quantiles      # CHANGED
        )

    def forward(
        self,
        x: torch.Tensor,
        edge_index: Adj,
        edge_weight: OptTensor = None,
        attn_mask: Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Args:
            x: [B, N, T, F_in] or [N, T, F_in]
            edge_index: [2, E]
            edge_weight: [E] optional edge weights
            attn_mask: [T, T] - Optional attention mask
            key_padding_mask: [B, T] - Optional padding mask

        Returns:
            out: [B, N, forecast_horizon] or [N, forecast_horizon]
        """
        batch_mode = x.dim() == 4
        if not batch_mode:
            x = x.unsqueeze(0)

        B, N, T, F = x.shape

        # 1) Spatial convolution per time-step using ChebConv
        # Reshape to process all batch-time combinations at once
        x_flat = x.permute(0, 2, 1, 3).reshape(B * T, N, F)  # [B*T, N, F]

        spatial_flat = []
        for i in range(B * T):
            # Apply ChebConv
            gconv_out = self.gconv(x_flat[i], edge_index, edge_weight)  # [N, g_hidden]

            # Optional residual connection
            res = self.residual(x_flat[i])  # [N, g_hidden]
            gconv_out = gconv_out + res

            # Project to spatial_channels
            spatial_out = self.spatial_proj(gconv_out)  # [N, spatial_channels]
            spatial_flat.append(spatial_out)

        spatial_flat = torch.stack(spatial_flat, dim=0)  # [B*T, N, spatial_channels]
        spatial_features = spatial_flat.view(B, T, N, self.spatial_channels).permute(0, 2, 1, 3)
        # [B, N, T, spatial_channels]

        # 2) Multi-head temporal attention with masking
        if self.temporal_attn is not None:
            attn_input = spatial_features.permute(0, 2, 1, 3)  # [B, T, N, spatial_channels]

            attn_output = self.temporal_attn(
                attn_input,
                attn_mask=attn_mask,
                key_padding_mask=key_padding_mask
            )  # [B, T, N, spatial_channels]

            attn_output = attn_output.permute(0, 2, 1, 3).contiguous()  # [B, N, T, spatial_channels]

            # Residual connection
            if self.attn_residual:
                spatial_features = spatial_features + attn_output
            else:
                spatial_features = attn_output

            # Layer normalization
            if self.norm_after_attn is not None:
                spatial_features = self.norm_after_attn(spatial_features)

        # 3) Temporal convolution
        # Reshape: [B, N, T, C] -> [B*N, C, T]
        C = spatial_features.size(-1)
        tcn_input = spatial_features.permute(0, 1, 3, 2).reshape(B * N, C, T)
        tcn_output = self.temporal_conv(tcn_input)  # [B*N, temporal_channels[-1], T]

        # 4) Take last timestep and project to forecast horizon
        tcn_last = tcn_output[:, :, -1]               # [B*N, temporal_channels[-1]]
        out = self.output_layer(tcn_last)              # [B*N, forecast_horizon * num_quantiles]
        out = out.view(B, N, self.forecast_horizon, self.num_quantiles)  # CHANGED

        if not batch_mode:
            out = out.squeeze(0)

        return out

quantiles = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]

n_out = 30

model = ChebConvTCN(
        in_channels=4,
        g_hidden=128,
        spatial_channels=128,
        temporal_channels=[128, 64],
        forecast_horizon=n_out,
        K=2,
        kernel_size=8,
        dropout=0.2,
        use_temporal_attention=True,
        num_heads=8,
        attn_dropout=0.2,
        attn_residual=True,
        use_layernorm_after_attn=True,
        causal_mask=True,
        quantiles=quantiles,
    )
print(model)

/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/libpyg.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sp

ChebConvTCN(
  (gconv): ChebConv(4, 128, K=2, normalization=sym)
  (spatial_proj): Linear(in_features=128, out_features=128, bias=True)
  (residual): Linear(in_features=4, out_features=128, bias=True)
  (temporal_attn): MultiHeadTemporalAttention(
    (query_proj): Linear(in_features=128, out_features=128, bias=True)
    (key_proj): Linear(in_features=128, out_features=128, bias=True)
    (value_proj): Linear(in_features=128, out_features=128, bias=True)
    (out_proj): Linear(in_features=128, out_features=128, bias=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (norm_after_attn): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (temporal_conv): TemporalConvNet(
    (network): Sequential(
      (0): Conv1d(128, 128, kernel_size=(8,), stride=(1,), padding=(7,))
      (1): Tanh()
      (2): Dropout(p=0.2, inplace=False)
      (3): Conv1d(128, 64, kernel_size=(8,), stride=(1,), padding=(14,), dilation=(2,))
      (4): Tanh()
      (5): Dropout(p=0.2, inplace=False)
    )


In [ ]:
import torch
import numpy as np
from pytorch_forecasting.metrics import QuantileLoss

# 1. Initialization
quantiles = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = QuantileLoss(quantiles=quantiles)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Ensure our graph variables are on the correct device
edge_index = edge_index.to(device)
edge_weight = edge_weight.to(device)

# 2. Training Function
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for X_batch, Y_batch in dataloader:
        # Move batches to GPU
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        optimizer.zero_grad()

        # Forward pass (Note: we pass the global edge_index/weight here)
        out = model(X_batch, edge_index=edge_index, edge_weight=edge_weight)
        # out shape: [Batch, Nodes, Horizon, Num_Quantiles]

        # Flatten Batch and Nodes so QuantileLoss can process it
        B, N, H, Q = out.shape
        preds = out.view(B * N, H, Q)     # Shape: [(Batch * Nodes), Horizon, Quantiles]
        targets = Y_batch.view(B * N, H)  # Shape: [(Batch * Nodes), Horizon]

        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


# 3. Validation Function
def val(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X_batch, Y_batch in dataloader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

            out = model(X_batch, edge_index=edge_index, edge_weight=edge_weight)

            B, N, H, Q = out.shape
            preds = out.view(B * N, H, Q)
            targets = Y_batch.view(B * N, H)

            loss = criterion(preds, targets)
            total_loss += loss.item()

    return total_loss / len(dataloader)


# 4. Testing Function
def test(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for X_batch, Y_batch in dataloader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

            out = model(X_batch, edge_index=edge_index, edge_weight=edge_weight)

            B, N, H, Q = out.shape
            preds = out.view(B * N, H, Q)
            targets = Y_batch.view(B * N, H)

            loss = criterion(preds, targets)
            total_loss += loss.item()

            # Store predictions and targets for later evaluation (keep original shape)
            all_predictions.append(out.cpu().numpy())
            all_targets.append(Y_batch.cpu().numpy())

    avg_loss = total_loss / len(dataloader)

    # Concatenate all batches together
    predictions = np.concatenate(all_predictions, axis=0) # [Total_Samples, Nodes, Horizon, Quantiles]
    targets = np.concatenate(all_targets, axis=0)         # [Total_Samples, Nodes, Horizon]

    return avg_loss, predictions, targets


# 5. Main Training Loop
epochs = 200 # Usually 2000 is too high without early stopping, adjust as needed!
best_val_loss = float('inf')

for epoch in range(epochs):
    # Notice we now pass `train_loader` instead of `train_windows`
    train_loss = train(model, train_loader, optimizer, criterion, device)

    if (epoch + 1) % 5 == 0:
        val_loss = val(model, val_loader, criterion, device)

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Loss:   {val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"  ✓ Saved best model (val loss: {val_loss:.6f})")

print("\n" + "="*50)
print("Final Evaluation on Unseen Test Data")
print("="*50)

# Load the best weights before running the test set!
model.load_state_dict(torch.load('best_model.pth'))

test_loss, test_preds, test_targets = test(model, test_loader, criterion, device)
print(f"Final Test Loss: {test_loss:.6f}")
print(f"Prediction shape: {test_preds.shape}") # [Samples, 54 Stores, 30 Days, 7 Quantiles]
print(f"Target shape: {test_targets.shape}")   # [Samples, 54 Stores, 30 Days]

Epoch 5/200
  Train Loss: 0.465319
  Val Loss:   0.555131
  ✓ Saved best model (val loss: 0.555131)
Epoch 10/200
  Train Loss: 0.351935
  Val Loss:   0.435785
  ✓ Saved best model (val loss: 0.435785)
Epoch 15/200
  Train Loss: 0.307519
  Val Loss:   0.395912
  ✓ Saved best model (val loss: 0.395912)
Epoch 20/200
  Train Loss: 0.281719
  Val Loss:   0.382571
  ✓ Saved best model (val loss: 0.382571)
Epoch 25/200
  Train Loss: 0.263600
  Val Loss:   0.383168
Epoch 30/200
  Train Loss: 0.253904
  Val Loss:   0.358933
  ✓ Saved best model (val loss: 0.358933)
Epoch 35/200
  Train Loss: 0.243277
  Val Loss:   0.335331
  ✓ Saved best model (val loss: 0.335331)
Epoch 40/200
  Train Loss: 0.235161
  Val Loss:   0.341712
Epoch 45/200
  Train Loss: 0.225959
  Val Loss:   0.322332
  ✓ Saved best model (val loss: 0.322332)
Epoch 50/200
  Train Loss: 0.217635
  Val Loss:   0.354307
Epoch 55/200
  Train Loss: 0.215941
  Val Loss:   0.364006
Epoch 60/200
  Train Loss: 0.212083
  Val Loss:   0.328412

In [ ]:
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.metrics import r2_score
import numpy as np

# 1. Inverse Transformation (Fixing Shape and Log1p)
test_preds_original = np.zeros(
    (test_preds.shape[0], test_preds.shape[1], test_preds.shape[2]),
    dtype=np.float32
)
test_targets_original = np.zeros_like(test_targets, dtype=np.float32)

for i in range(test_preds.shape[2]): # Loop over horizon (n_out)

    # Extract median quantile (0.5 is index 3 in our 7-quantile list)
    if test_preds.ndim == 4:
        q_idx = 3 # Index for the 0.5 (median) quantile
        pred_step = test_preds[:, :, i, q_idx]   # [num_windows, N]
    else:
        pred_step = test_preds[:, :, i]

    target_step = test_targets[:, :, i]

    # A. Reshape to (-1, 1), inverse standard scale, reshape back
    inv_pred_scaled = scaler.inverse_transform(pred_step.reshape(-1, 1)).reshape(pred_step.shape)
    inv_target_scaled = scaler.inverse_transform(target_step.reshape(-1, 1)).reshape(target_step.shape)

    # B. Reverse the log1p transformation to get actual sales quantities
    test_preds_original[:, :, i] = np.expm1(inv_pred_scaled)
    test_targets_original[:, :, i] = np.expm1(inv_target_scaled)

# 2. Extract 1-Step Ahead Forecast for Visualization & Metrics
inversed_actuals = test_targets_original[:, :, 0]    # [num_windows, N]
inversed_predictions = test_preds_original[:, :, 0]  # [num_windows, N]

# 3. Visualization
plt.figure(figsize=(16, 12))
plt.suptitle('ChebConvTCN Predictions vs Actuals (1-Day Ahead)', fontsize=18, y=1.02)

# Plotting the first 15 stores to avoid overcrowding the plot
num_stores_to_plot = min(15, inversed_actuals.shape[1])

for i in range(num_stores_to_plot):
    plt.subplot(5, 3, i+1)
    plt.plot(inversed_actuals[:, i], label='Actual', color='blue')
    plt.plot(inversed_predictions[:, i], label='Predicted', color='red', linestyle='--')
    plt.title(f'Store {i+1} Sales')
    plt.xlabel('Time Step')
    plt.ylabel('Unit Sales')
    plt.legend()
    plt.grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# 4. Safe Metrics Functions
def mean_absolute_percentage_error(y_test, y_pred):
    y_true, y_pred = np.array(y_test), np.array(y_pred)
    # Add epsilon to prevent division by zero on days with 0 sales
    epsilon = 1e-8
    return np.mean(np.abs((y_test - y_pred) / (y_test + epsilon))) * 100

def mean_squared_prediction_error(y_test, y_pred):
    y_true, y_pred = np.array(y_test), np.array(y_pred)
    return np.mean(((y_test - y_pred))**2)

# 5. Evaluate Overall Metrics across all stores
print("\n" + "="*40)
print("Overall Metrics (1-Day Ahead Forecast)")
print("="*40)
print(f'MSE:       {metrics.mean_squared_error(inversed_actuals, inversed_predictions):.4f}')
print(f'MAE:       {metrics.mean_absolute_error(inversed_actuals, inversed_predictions):.4f}')
print(f'RMSE:      {np.sqrt(metrics.mean_squared_error(inversed_actuals, inversed_predictions)):.4f}')
print(f'MAPE:      {mean_absolute_percentage_error(inversed_actuals, inversed_predictions):.4f}%')
print(f'MSPE:      {mean_squared_prediction_error(inversed_actuals, inversed_predictions):.4f}')
print(f'sqrt MSPE: {np.sqrt(mean_squared_prediction_error(inversed_actuals, inversed_predictions)):.4f}')
print(f'R2 Score:  {r2_score(inversed_actuals, inversed_predictions):.4f}')

In [ ]:
def numpy_normalised_quantile_loss(y, y_pred, quantile):
  """Computes normalised quantile loss for numpy arrays.

  Uses the q-Risk metric as defined in the "Training Procedure" section of the
  main TFT paper.

  Args:
    y: Targets
    y_pred: Predictions
    quantile: Quantile to use for loss calculations (between 0 & 1)

  Returns:
    Float for normalised quantile loss.
  """
  prediction_underflow = y - y_pred
  weighted_errors = quantile * np.maximum(prediction_underflow, 0.) \
      + (1. - quantile) * np.maximum(-prediction_underflow, 0.)

  quantile_loss = weighted_errors.mean()
  normaliser = np.abs(y).mean() # Corrected line: use np.abs for numpy arrays

  return 2 * quantile_loss / normaliser

# Assuming your quantiles list is: [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
p50_idx = quantiles.index(0.5)
p90_idx = quantiles.index(0.9)

# 1. Extract the specific quantile predictions across the FULL 30-day horizon
# test_preds shape is [Samples, Stores, Horizon, Quantiles]
preds_p50 = test_preds[:, :, :, p50_idx]
preds_p90 = test_preds[:, :, :, p90_idx]
targets_full = test_targets

# 2. Inverse transform the full 3D arrays
# (Flatten to 2D for the scaler, then reshape back to 3D)
inv_preds_p50 = scaler.inverse_transform(preds_p50.reshape(-1, 1)).reshape(preds_p50.shape)
inv_preds_p90 = scaler.inverse_transform(preds_p90.reshape(-1, 1)).reshape(preds_p90.shape)
inv_targets = scaler.inverse_transform(targets_full.reshape(-1, 1)).reshape(targets_full.shape)

# 3. Reverse the log1p transformation to get real unit sales
preds_p50_real = np.expm1(inv_preds_p50)
preds_p90_real = np.expm1(inv_preds_p90)
actuals_real = np.expm1(inv_targets)

# 4. Calculate the q-Risk using your custom function
p50_risk = numpy_normalised_quantile_loss(actuals_real, preds_p50_real, 0.5)
p90_risk = numpy_normalised_quantile_loss(actuals_real, preds_p90_real, 0.9)

print(f"P50 q-Risk: {p50_risk:.4f}")
print(f"P90 q-Risk: {p90_risk:.4f}")

In [ ]:
from google.colab import runtime

# This will disconnect and delete the current runtime
print("Training complete. Disconnecting runtime...")
runtime.unassign()
